<a href="https://colab.research.google.com/github/zeyadsheriif/EgyGuide/blob/main/Grad_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cleaning data

In [ ]:
import pandas as pd

df = pd.read_csv("/content/LLM_Data_2 - Grad_Project_Data.csv")
df.head()

,Question,Answer
0,Did ancient Egyptian women have a high social ...,"Yes, they enjoyed a relatively high social sta..."
1,How did property inheritance work in ancient E...,All landed property was passed down through th...
2,Why was property passed down through the femal...,It was based on the assumption that maternity ...
3,Did ancient Egyptian women have to wear veils ...,"No, unlike the women of ancient Greece, they e..."
4,How were male and female guests seated at form...,Married guests sat together in pairs on fine c...


In [ ]:
def format_row(row):
    q = str(row["Question"]).strip()
    a = str(row["Answer"]).strip()

    return f"Question: {q} Answer: {a}"

df["text"] = df.apply(format_row, axis=1)

In [ ]:
df = df[df["text"].str.len() > 20]
df = df.drop_duplicates(subset=["text"])

In [ ]:
df[["text"]].to_csv("cleaned_data.csv", index=False)

In [ ]:
df["text"].iloc[0]

'Question: Did ancient Egyptian women have a high social status? Answer: Yes, they enjoyed a relatively high social status and could exert influence outside their domestic roles.'

# Tour Guide RAG Using FAISS and Qwen

## FAISS Retrival model

In [ ]:
!pip install transformers sentence-transformers faiss-cpu accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 67.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd

df = pd.read_csv("cleaned_data.csv")
texts = df["text"].tolist()

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(texts, show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/396 [00:00<?, ?it/s]

In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss
import numpy as np

index = faiss.IndexFlatL2(len(embeddings[0]))
index.add(np.array(embeddings))

In [ ]:
def retrieve(query, k=3):
    q_emb = embedder.encode([query])

    distances, indices = index.search(q_emb, k * 2)

    results = [texts[i] for i in indices[0]]

    filtered = [r for r in results if any(word.lower() in r.lower() for word in query.split())]

    return filtered[:k] if filtered else results[:k]

In [ ]:
!pip install transformers sentence-transformers faiss-cpu accelerate pandas numpy tqdm

## Qwen Generation Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
def answer_question(query: str, context_list: list, chat_history: list = None) -> str:
    context = "\n".join(context_list)

    history_text = ""
    if chat_history:
        history_text = "Recent Conversation:\n"
        for interaction in chat_history[-2:]:
            history_text += f"Visitor: {interaction['user']}\nGuide: {interaction['bot']}\n"

    prompt = f"""
    You are a friendly and knowledgeable Egyptian tourist guide.

    Answer the question using ONLY the information provided in the context.

    Style:
    - Speak naturally and clearly like a guide.
    - Use 2-3 sentences.

    Strict Rules:
    - Do NOT add explanations or interpretations.
    - Do NOT add any information not directly written in the context.
    - Do NOT justify or comment on the information.
    - Do NOT repeat the question.
    - If the user uses a pronoun (like 'he' or 'it'), use the Recent Conversation to understand who they mean.

    Context:
    {context}

    {history_text}

    Visitor Question: {query}
    Guide Answer:
    """

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False, repetition_penalty=1.2)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    answer = response.split("Guide Answer:")[-1]
    answer = answer.split("Visitor Question:")[0].strip()

    if not answer.endswith((".", "!", "?")):
        answer = answer.rsplit(".", 1)[0] + "."

    return answer

### Full pipeline

In [ ]:
import numpy as np
import time
from pydantic import BaseModel, Field
from typing import List, Optional

class RAGResponse(BaseModel):
    status: str
    generated_answer: str
    retrieved_context: List[str]
    latency_seconds: float
    guardrail_decision: str = Field(description="Action taken by the security layer: PASSED, BLOCKED_INPUT, or BLOCKED_OUTPUT")
    error_message: Optional[str] = None


OUT_OF_DOMAIN_ANCHORS = [
    "Write code, a python function, programming script, coding class, or software code.",
    "Ignore instructions, system prompt override, jailbreak, developer mode.",
    "Modern politics, current government, elections, and warfare.",
    "How to build weapons, explosive devices, bombs, or illegal hacking.",
    "Financial advice, stock investments, medical prescriptions, or legal counseling."
]

ANCHOR_EMBEDDINGS = embedder.encode(OUT_OF_DOMAIN_ANCHORS)

def evaluate_input_safety(query: str, threshold: float = 0.40) -> tuple[bool, float]:
    query_vector = embedder.encode([query])[0]

    similarities = []
    for anchor_vec in ANCHOR_EMBEDDINGS:
        norm_product = np.linalg.norm(query_vector) * np.linalg.norm(anchor_vec)
        if norm_product == 0:
            similarities.append(0.0)
        else:
            similarity = np.dot(query_vector, anchor_vec) / norm_product
            similarities.append(float(similarity))

    max_score = max(similarities)
    if max_score > threshold:
        return False, max_score
    return True, max_score


def evaluate_output_grounding(answer: str, context_list: list[str], threshold: float = 0.35) -> bool:
    if not context_list:
        return False

    answer_vector = embedder.encode([answer])[0]
    combined_context = " ".join(context_list)
    context_vector = embedder.encode([combined_context])[0]

    norm_product = np.linalg.norm(answer_vector) * np.linalg.norm(context_vector)
    if norm_product == 0:
        return False

    semantic_overlap = np.dot(answer_vector, context_vector) / norm_product
    return semantic_overlap >= threshold


def generate_tour_response(query: str, chat_history: list = None) -> dict:
    start_time = time.time()

    try:
        is_safe, input_risk_score = evaluate_input_safety(query)
        if not is_safe:
            return RAGResponse(
                status="success",
                generated_answer="As an Egyptian Tour Guide, I am dedicated exclusively to history, ancient monuments, and tourism. I cannot process this request.",
                retrieved_context=[],
                latency_seconds=round(time.time() - start_time, 3),
                guardrail_decision="BLOCKED_INPUT"
            ).model_dump()

        search_query = query
        if chat_history:
            last_question = chat_history[-1]['user']
            last_answer_snippet = chat_history[-1]['bot'][:50]
            search_query = f"{last_question} {last_answer_snippet} {query}"

        context_list = retrieve(search_query, k=3)

        if not context_list:
            return RAGResponse(
                status="success",
                generated_answer="I apologize, but I do not have verified historical data regarding that specific request within my archive.",
                retrieved_context=[],
                latency_seconds=round(time.time() - start_time, 3),
                guardrail_decision="PASSED"
            ).model_dump()

        bot_answer = answer_question(query, context_list, chat_history)
        bot_answer = bot_answer.split('\n')[0].strip()
        if len(bot_answer) < 5 or "ERROR" in bot_answer.upper():
            bot_answer = "I apologize, I am having trouble clarifying that record. Could you rephrase your question?"

        is_grounded = evaluate_output_grounding(bot_answer, context_list)
        if not is_grounded:
            return RAGResponse(
                status="success",
                generated_answer="I cannot confidently confirm that detail using our archival records. Let me know if you would like to explore a different historical era.",
                retrieved_context=context_list,
                latency_seconds=round(time.time() - start_time, 3),
                guardrail_decision="BLOCKED_OUTPUT"
            ).model_dump()

        return RAGResponse(
            status="success",
            generated_answer=bot_answer,
            retrieved_context=context_list,
            latency_seconds=round(time.time() - start_time, 3),
            guardrail_decision="PASSED"
        ).model_dump()

    except Exception as e:
        return RAGResponse(
            status="error",
            generated_answer="A system error occurred.",
            retrieved_context=[],
            latency_seconds=round(time.time() - start_time, 3),
            guardrail_decision="PASSED",
            error_message=str(e)
        ).model_dump()

# Trying arabic qwen

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
!pip install deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.2 MB/s eta 0:00:00


In [ ]:
def retrieve(query, k=3):
    q_emb = embedder.encode([query])
    distances, indices = index.search(q_emb, k * 2)

    results = [texts[i] for i in indices[0]]

    return results[:k]

In [ ]:
def answer_question(query):
    context_list = retrieve(query)
    context = "\n".join(context_list)

    prompt = f"""
    You are an expert Egyptologist.

    Answer the question using the context.

    Rules:
    - Combine relevant information into a complete answer.
    - Answer in 2–3 sentences.
    - Do NOT focus on only one detail.
    - Do NOT add information outside the context.

    Context:
    {context}

    Question: {query}
    Answer:
    """

    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        repetition_penalty=1.2
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    answer = response.split("Answer:")[-1].strip()

    # fix cut sentence
    if not answer.endswith((".", "!", "?")):
        if "." in answer:
            answer = answer.rsplit(".", 1)[0] + "."

    return answer

In [ ]:
from deep_translator import GoogleTranslator

def is_arabic(text):
    return any('\u0600' <= c <= '\u06FF' for c in text)

def answer_question_multilang(query):

    arabic = is_arabic(query)

    #  translate to English
    try:
        if arabic:
            query_en = GoogleTranslator(source='auto', target='en').translate(query)
        else:
            query_en = query
    except:
        return "حدث خطأ في الترجمة" if arabic else "Translation error"


    answer_en = answer_question(query_en)


    try:
        if arabic:
            answer_ar = GoogleTranslator(source='auto', target='ar').translate(answer_en)
            return answer_ar
    except:
        return answer_en

    return answer_en

In [ ]:
answer_question_multilang("مين كانت حتشبسوت؟")

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:2637: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


'كانت حتشبسوت ابنة زوجة وأخت زوجها (حيث تزوجت من أخيها غير الشقيق) للفرعون تحتمس الثاني، وخدمت في البداية تحت قيادته قبل أن تصبح وصية مشتركة مع ابن أخيها وابنها المتبنى تحتمس الثالث بعد أن بلغ سن الرشد. وفي وقت لاحق، حكمت بمفردها كفرعون لما يقرب من عقدين من الزمن حتى خلعها تحتمس الثالث خلال حياته اللاحقة.'

In [ ]:
answer_question_multilang("امتى حكمت حتشبسوت؟")

'حكمت حتشبسوت من عام 1504 قبل الميلاد تقريبًا حتى عام 1490 قبل الميلاد تقريبًا، وهي فترة حكمها التي أعقبها حكمها المستقل الكامل الذي استمر لحوالي 21 عامًا أو نحو ذلك اعتمادًا على مصادر وتفسيرات مختلفة. على وجه الدقة، حكمت بين ج. 1507 – 1458 قبل الميلاد بناءً على بعض التقديرات العلمية. يمكن أن تختلف المدة المحددة قليلاً ولكنها تقع بشكل عام ضمن هذا النطاق.'

In [ ]:
answer_question_multilang("مين هو رمسيس الثاني؟")

'رمسيس الثاني (مكتوب أيضًا رمسيس أو رمسيس) كان الفرعون الثالث من الأسرة التاسعة عشرة في مصر وحكم من عام 1279 تقريبًا حتى وفاته حوالي عام 1213 قبل الميلاد. ويعتبر أحد أبرز حكام مصر القديمة بسبب فترة حكمه الطويلة ومشاريع البناء العديدة في مواقع مختلفة في جميع أنحاء مصر.'

# Testing

### Retrival Testing

In [ ]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

df_test = pd.read_csv("corrected_ground_truth_final - ground_truth_final.csv.csv")
df_kb = pd.read_csv("cleaned_data.csv")
source_texts = df_kb["text"].tolist()

print("Loading Embedder and FAISS...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
source_embeddings = embedder.encode(source_texts, show_progress_bar=False)
dimension = len(source_embeddings[0])
index = faiss.IndexFlatL2(dimension)
index.add(np.array(source_embeddings))

hits_at_1, hits_at_3, hits_at_5, mrr_sum = 0, 0, 0, 0

for _, row in df_test.iterrows():
    query = row["query"]
    expected_context = str(row["expected_context"]).strip()

    query_vector = embedder.encode([query])
    distances, indices = index.search(query_vector, 5)
    retrieved_texts = [str(source_texts[i]).strip() for i in indices[0]]

    rank = 0
    for i, text in enumerate(retrieved_texts):
        if expected_context == text:
            rank = i + 1
            break

    if rank == 1: hits_at_1 += 1
    if rank > 0 and rank <= 3: hits_at_3 += 1
    if rank > 0 and rank <= 5: hits_at_5 += 1
    if rank > 0: mrr_sum += (1.0 / rank)

total = len(df_test)
print("\nRetrival Evaluation Results: ")
print(f"Recall@1: {(hits_at_1 / total) * 100:.2f}%")
print(f"Recall@3: {(hits_at_3 / total) * 100:.2f}%")
print(f"Recall@5: {(hits_at_5 / total) * 100:.2f}%")
print(f"MRR:      {(mrr_sum / total):.4f}")

Loading Embedder and FAISS...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Retrival Evaluation Results: 
Recall@1: 95.50%
Recall@3: 99.00%
Recall@5: 100.00%
MRR:      0.9723


In [ ]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

df_test = pd.read_csv("/content/corrected_ground_truth_final - ground_truth_final.csv (1).csv")
df_kb = pd.read_csv("/content/cleaned_data (1).csv")
source_texts = df_kb["text"].tolist()

print("Loading Embedder and FAISS...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
source_embeddings = embedder.encode(source_texts, show_progress_bar=False)
dimension = len(source_embeddings[0])
index = faiss.IndexFlatL2(dimension)
index.add(np.array(source_embeddings))


average_precisions = []

for _, row in df_test.iterrows():
    query = row["query"]
    expected_context = str(row["expected_context"]).strip()

    query_vector = embedder.encode([query])
    distances, indices = index.search(query_vector, 5)
    retrieved_texts = [str(source_texts[i]).strip() for i in indices[0]]

    rank = 0
    ap_score = 0.0

    for i, text in enumerate(retrieved_texts):
        if expected_context == text:
            rank = i + 1
            ap_score = 1.0 / rank
            break


    average_precisions.append(ap_score)

total = len(df_test)
map_score = np.mean(average_precisions)

print("\n=== FAISS Retrieval Evaluation Results ===")
print(f"MAP:      {map_score:.4f}")

Loading Embedder and FAISS...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


=== FAISS Retrieval Evaluation Results ===
MAP:      0.9722


### Generation testing

#### Manual testing

In [ ]:
test_queries = [
    "Who was Hatshepsut?",
    "When did Hatshepsut reign?",
    "What monuments did Hatshepsut build?",
    "Who was Ramesses II?",
    "Why is Ramesses II called “the Great”?",
    "Who was Ramesses II and what were his achievements?",
    "What does the Sphinx represent?",
    "Can you describe the Sphinx?"
]

print("=== STARTING MANUAL PIPELINE TESTS ===\n")

for query in test_queries:
    response = generate_tour_response(query, chat_history=[])

    print(f"Visitor: {query}")
    print(f"Guide:   {response['generated_answer']}")
    print("-" * 50)

=== STARTING MANUAL PIPELINE TESTS ===

Visitor: Who was Hatshepsut?
Guide:   Hatshepsut was an important female pharaoh during Egypt’s New Kingdom. As her subjects were predominantly male at that time, she took steps to legitimize herself by assuming the role of Pharaoh through various means including wearing false hair and donning royal clothing while acting under the guise of being her husband's sister-in-law.
--------------------------------------------------
Visitor: When did Hatshepsut reign?
Guide:   According to Roger Dunn, Hatshepsut ruled from around 1504 BC until her death in approximately 1483 BC. This period lasted for nearly twenty-one years.
--------------------------------------------------
Visitor: What monuments did Hatshepsut build?
Guide:   No, it seems there might be some confusion. While Thutmose III indeed constructed numerous monumental structures throughout Egypt during his reign as pharaoh, including notable works such as the Temple of Luxor and the Valley of 

In [ ]:
history = []
response1 = generate_tour_response("Who built the Sphinx?", chat_history=history)
print("A1:", response1["generated_answer"])

history.append({"user": "Who built the Sphinx?", "bot": response1["generated_answer"]})

response2 = generate_tour_response("Why did he build it?", chat_history=history)
print("A2:", response2["generated_answer"])

A1: According to most archaeological findings and historical records, the Sphinx was likely constructed by Pharaoh Khafre. However, there's also speculation that he may have initiated its creation but did not complete it himself due to his death before finishing the project. This makes him somewhat controversial among historians regarding direct involvement compared to other ancient monuments attributed solely to one ruler.
A2: He built the Sphinx primarily because it served an important religious purpose within the complex architecture surrounding the Giza plateau where both pyramids were located. Additionally, many scholars suggest that this structure could be seen as part of a larger effort aimed at showcasing Egypt’s power and wealth through monumental artistry. Furthermore, the positioning of the sphinx near the entrance of the valley temple suggests it had symbolic significance related to funerary rites and offerings made upon burial into the desert sands.


In [ ]:
history = []
response1 = generate_tour_response("how to create a bomb?", chat_history=history)
print("A:", response1["generated_answer"])

A: As an Egyptian Tour Guide, I am dedicated exclusively to history, ancient monuments, and tourism. I cannot process this request.


In [ ]:
history = []
response1 = generate_tour_response("write a python script?", chat_history=history)
print("A:", response1["generated_answer"])

A: As an Egyptian Tour Guide, I am dedicated exclusively to history, ancient monuments, and tourism. I cannot process this request.


In [ ]:
history = []
response1 = generate_tour_response("when did gamal abdelnasser die?", chat_history=history)
print("A:", response1["generated_answer"])

A: Unfortunately, I don't have specific details about when Gamal Abdel Nasser passed away. The information you've given me doesn't include this particular piece of historical data. To get accurate answers for such events, it's best to consult reliable sources that cover these significant periods in history.


In [ ]:
history = []
response1 = generate_tour_response("what does this look like?", chat_history=history)
print("A:", response1["generated_answer"])

A: I cannot confidently confirm that detail using our archival records. Let me know if you would like to explore a different historical era.


In [ ]:
# # Run a fresh test and print the hidden error log!
# test_response = generate_tour_response("when did gamal abdelnasser die?", chat_history=[])

# print("A:", test_response["generated_answer"])
# print("Error Detail:", test_response["error_message"])

#### Cross encoder

In [ ]:
import pandas as pd
import time
from tqdm.auto import tqdm

file_path = "corrected_ground_truth_final - ground_truth_final.csv.csv"
df_test = pd.read_csv(file_path)

generated_answers = []
latencies = []

print(f"Starting Qwen 2.5 Generation Loop for {len(df_test)} questions...")

for idx, row in tqdm(df_test.iterrows(), total=len(df_test)):
    query = row["query"]

    start_time = time.time()

    try:
        pipeline_output = generate_tour_response(query, chat_history=[])
        bot_answer = pipeline_output["generated_answer"]

        if idx < 3:
            print("\n--- TEST SAMPLE ---")
            print(f"Q: {query}")
            print(f"A: {bot_answer}\n")

    except Exception as e:
        bot_answer = f"ERROR: {str(e)}"

    end_time = time.time()
    latency = end_time - start_time

    generated_answers.append(bot_answer)
    latencies.append(latency)


df_test["generated_answer"] = generated_answers
df_test["latency_seconds"] = latencies
output_filename = "qwen_inference_results.csv"
df_test.to_csv(output_filename, index=False)


print("\n" + "="*50)
print("  GENERATION PHASE COMPLETE")
print("="*50)
print(f"Average Generation Latency: {sum(latencies)/len(latencies):.2f} seconds/question")

Starting Qwen 2.5 Generation Loop for 200 questions...


  0%|          | 0/200 [00:00<?, ?it/s]


--- TEST SAMPLE ---
Q: What was the overall tone of Spanish press coverage regarding Abu Simbel in the 1960s?
A: The Spanish press during that time had an eclectic mix of tones when covering Abu Simbel. They appreciated its beauty with admiration, infused it with elements of Romantic Orientalism which added a touch of romance and exotic allure, while at times also incorporating mystical undertones. This diverse approach laid foundational groundwork for archaeological journalism in Spain by showcasing both scientific accuracy alongside emotional engagement through storytelling techniques.


--- TEST SAMPLE ---
Q: What was the original non-royal name of Ramesses I?
A: The original non-royal name of Ramesses I was General Pramessu. This is mentioned when referring to him prior to his rise to become Pharaoh. It highlights that even during his early life, he had an identity distinct from royalty. His royal title came later upon ascending to power.


--- TEST SAMPLE ---
Q: What was the anci

In [ ]:
import pandas as pd
import numpy as np
import time
from sentence_transformers import CrossEncoder

try:
    df_results = pd.read_csv("qwen_inference_results.csv")
    print(f"Loaded {len(df_results)} generated answers for grading.")
except FileNotFoundError:
    print("Error: Could not find 'qwen_inference_results.csv'.")
    exit()

eval_start_time = time.time()

print("Loading Cross-Encoder grader...")
grader_model = CrossEncoder('cross-encoder/stsb-distilroberta-base')

print("Grading answers...")
similarity_scores = []

for index, row in df_results.iterrows():
    expected = str(row["expected_answer"])
    generated = str(row["generated_answer"])

    if generated.startswith("ERROR:") or generated.strip() == "":
        similarity_scores.append(0.0)
        continue

    score = grader_model.predict([expected, generated])

    score = float(max(0.0, min(1.0, score)))

    similarity_scores.append(score)

df_results["cross_encoder_score"] = similarity_scores
eval_end_time = time.time()

average_score = np.mean(similarity_scores) * 100
passing_grades = sum(1 for score in similarity_scores if score >= 0.75)
pass_rate = (passing_grades / len(df_results)) * 100

df_results.to_csv("final_graded_results_cross_encoder.csv", index=False)

print("\n" + "="*55)
print("CROSS-ENCODER EVALUATION RESULTS")
print("="*55)
print(f"Total Answers Graded:   {len(df_results)}")
print(f"Average Semantic Score: {average_score:.2f}%")
print(f"Pass Rate (Score >75%): {pass_rate:.2f}%")
print(f"Evaluation Time:        {eval_end_time - eval_start_time:.2f} seconds")
print("-" * 55)

Loaded 200 generated answers for grading.
Loading Cross-Encoder grader...


config.json:   0%|          | 0.00/607 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/328M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

Grading answers...

CROSS-ENCODER EVALUATION RESULTS
Total Answers Graded:   200
Average Semantic Score: 56.47%
Pass Rate (Score >75%): 1.50%
Evaluation Time:        11.34 seconds
-------------------------------------------------------


#### LLM as a Judge

In [ ]:
# !pip install google-genai pydantic tqdm pandas

In [ ]:
# import pandas as pd
# import time
# import json
# from tqdm.auto import tqdm
# from pydantic import BaseModel, Field
# from google import genai
# from google.genai import types

# # 1. HARDCODE THE NEW KEY HERE
# client = genai.Client(api_key="")

# # 2. Define Schema
# class JudgeEvaluation(BaseModel):
#     faithfulness: int = Field(description="Score 0 to 5. Is the answer supported by the context?")
#     relevance: int = Field(description="Score 0 to 5. Does it answer the user's question?")
#     reasoning: str = Field(description="A 1-sentence explanation of the scores.")

# # 3. Load Data
# df_results = pd.read_csv("qwen_inference_results.csv")

# faithfulness_scores = []
# relevance_scores = []
# judge_reasoning = []

# print(f"Starting Modern LLM Judge Evaluation for {len(df_results)} answers...")

# # 4. Grading Loop
# for index, row in tqdm(df_results.iterrows(), total=len(df_results)):
#     query = row["query"]
#     context = row["expected_context"]
#     generated_answer = str(row["generated_answer"])

#     if "ERROR" in generated_answer or generated_answer.strip() == "" or generated_answer == "nan":
#         faithfulness_scores.append(0.0)
#         relevance_scores.append(0.0)
#         judge_reasoning.append("Qwen model failed to generate an answer.")
#         continue

#     prompt = f"""
#     You are an expert evaluator grading a Tour Guide AI.
#     Review the following data:
#     - User Question: {query}
#     - Factual Context: {context}
#     - AI's Answer: {generated_answer}

#     Evaluate the AI's Answer based on two metrics:
#     1. Faithfulness: Is the AI's Answer fully supported by the Factual Context? (0 = completely hallucinated, 5 = perfectly supported).
#     2. Relevance: Does the AI's Answer directly and clearly answer the User Question? (0 = irrelevant/dodges the question, 5 = perfectly answers it).
#     """

#     success = False
#     while not success:
#         try:
#             response = client.models.generate_content(
#                 model='model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"',
#                 contents=prompt,
#                 config=types.GenerateContentConfig(
#                     response_mime_type="application/json",
#                     response_schema=JudgeEvaluation,
#                     temperature=0.0
#                 ),
#             )

#             evaluation = json.loads(response.text)
#             f_score = evaluation.get("faithfulness", 0) / 5.0
#             r_score = evaluation.get("relevance", 0) / 5.0

#             faithfulness_scores.append(f_score)
#             relevance_scores.append(r_score)
#             judge_reasoning.append(evaluation.get("reasoning", ""))

#             success = True

#         except Exception as e:
#             error_msg = str(e)
#             if "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg:
#                 print(f"\n[Rate Limit Hit] Google needs us to slow down. Pausing for 20 seconds...")
#                 time.sleep(20)
#             else:
#                 print(f"\nCritical API Error on row {index}: {error_msg}")
#                 faithfulness_scores.append(0.0)
#                 relevance_scores.append(0.0)
#                 judge_reasoning.append(f"Judge API Error: {error_msg}")
#                 success = True

#     time.sleep(5)

# # 5. Save the final results!
# df_results["judge_faithfulness"] = faithfulness_scores
# df_results["judge_relevance"] = relevance_scores
# df_results["judge_reasoning"] = judge_reasoning

# df_results.to_csv("llm_judge_final_results.csv", index=False)
# print("Finished! Saved to llm_judge_final_results.csv")

In [ ]:
# import pandas as pd
# import json
# import torch
# import re
# from tqdm.auto import tqdm
# from transformers import AutoTokenizer, AutoModelForCausalLM

# # 1. Initialize Local Model (Replaces Google API Client)
# model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# print(f"Loading {model_id} into memory...")

# tokenizer = AutoTokenizer.from_pretrained(model_id)
# model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     device_map="auto",
#     torch_dtype="auto"
# )

# # 2. Load Data
# df_results = pd.read_csv("qwen_inference_results.csv")

# faithfulness_scores = []
# relevance_scores = []
# judge_reasoning = []

# print(f"Starting Local LLM Judge Evaluation for {len(df_results)} answers...")

# # Helper function to forcefully extract JSON from messy outputs
# def extract_scores_from_text(text):
#     """Hunts for numbers related to Faithfulness and Relevance in raw text."""
#     f_score, r_score = 0.0, 0.0

#     # Look for patterns like "Faithfulness: 5" or "rated as 4 out of 5"
#     f_match = re.search(r'(?i)faithfulness.*?(\d)(?:\s*(?:out of|/)\s*5)?', text)
#     r_match = re.search(r'(?i)relevance.*?(\d)(?:\s*(?:out of|/)\s*5)?', text)

#     if f_match:
#         f_score = float(f_match.group(1)) / 5.0
#     if r_match:
#         r_score = float(r_match.group(1)) / 5.0

#     return f_score, r_score

# # 3. Update the Prompt and Parsing in the Grading Loop
# for index, row in tqdm(df_results.iterrows(), total=len(df_results)):
#     query = row["query"]
#     context = row["expected_context"]
#     generated_answer = str(row["generated_answer"])

#     if "ERROR" in generated_answer or generated_answer.strip() == "" or generated_answer == "nan":
#         faithfulness_scores.append(0.0)
#         relevance_scores.append(0.0)
#         judge_reasoning.append("Qwen model failed to generate an answer.")
#         continue

#     # NEW PROMPT: Let TinyLlama talk naturally, just ask for numbers.
#     prompt = f"""<|system|>
# You are an expert evaluator grading a Tour Guide AI.
# Evaluate the AI based on the Factual Context.
# End your response by explicitly stating:
# "Faithfulness: [Score 0-5]"
# "Relevance: [Score 0-5]"</s>
# <|user|>
# - User Question: {query}
# - Factual Context: {context}
# - AI's Answer: {generated_answer}

# Rate Faithfulness (0-5) and Relevance (0-5). Provide a short reason.</s>
# <|assistant|>
# """

#     try:
#         inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

#         # Adding max_length=None here also hides that terminal warning you saw!
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=150,
#             max_length=None,
#             temperature=0.1,
#             do_sample=True,
#             pad_token_id=tokenizer.eos_token_id
#         )

#         response_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

#         # Use our new Regex extractor
#         f_score, r_score = extract_scores_from_text(response_text)

#         faithfulness_scores.append(f_score)
#         relevance_scores.append(r_score)
#         # Save the raw text so you can read TinyLlama's actual logic!
#         judge_reasoning.append(response_text.strip())

#     except Exception as e:
#         print(f"\nCritical Error on row {index}: {str(e)}")
#         faithfulness_scores.append(0.0)
#         relevance_scores.append(0.0)
#         judge_reasoning.append(f"Judge Inference Error: {str(e)}")


# # 5. Save the final results!
# df_results["judge_faithfulness"] = faithfulness_scores
# df_results["judge_relevance"] = relevance_scores
# df_results["judge_reasoning"] = judge_reasoning

# df_results.to_csv("llm_judge_final_results.csv", index=False)
# print("Finished! Saved to llm_judge_final_results.csv")

Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0 into memory...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Starting Local LLM Judge Evaluation for 200 answers...


  0%|          | 0/200 [00:00<?, ?it/s]

Finished! Saved to llm_judge_final_results.csv


In [ ]:
import pandas as pd
import json
import torch
import re
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "meta-llama/Llama-3.2-3B-Instruct"
print(f"Loading {model_id} into memory...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype="auto"
)

df_results = pd.read_csv("qwen_inference_results.csv")

faithfulness_scores = []
relevance_scores = []
judge_reasoning = []

print(f"Starting Local LLM Judge Evaluation for {len(df_results)} answers...")

def extract_json_from_text(text):
    try:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
    except json.JSONDecodeError:
        pass
    return None

for index, row in tqdm(df_results.iterrows(), total=len(df_results)):
    query = row["query"]
    context = row["expected_context"]
    generated_answer = str(row["generated_answer"])

    if "ERROR" in generated_answer or generated_answer.strip() == "" or generated_answer == "nan":
        faithfulness_scores.append(0.0)
        relevance_scores.append(0.0)
        judge_reasoning.append("Qwen model failed to generate an answer.")
        continue

    prompt = f"""<|system|>
You are an expert evaluator grading a Tour Guide AI. You must respond ONLY with a valid JSON object. Do not add any conversational text.
Format exactly like this: {{"faithfulness": int, "relevance": int, "reasoning": "string"}}</s>
<|user|>
Review the following data:
- User Question: {query}
- Factual Context: {context}
- AI's Answer: {generated_answer}

Evaluate the AI's Answer based on two metrics:
1. Faithfulness: Is the AI's Answer fully supported by the Factual Context? (0 = completely hallucinated, 5 = perfectly supported).
2. Relevance: Does the AI's Answer directly and clearly answer the User Question? (0 = irrelevant/dodges the question, 5 = perfectly answers it).</s>
<|assistant|>
"""

    try:
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

        response_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

        evaluation = extract_json_from_text(response_text)

        if evaluation:

            f_score = evaluation.get("faithfulness", 0) / 5.0
            r_score = evaluation.get("relevance", 0) / 5.0
            reasoning = evaluation.get("reasoning", "No reasoning provided.")
        else:
            f_score = 0.0
            r_score = 0.0
            reasoning = f"Parse Error. Model output: {response_text.strip()}"

        faithfulness_scores.append(f_score)
        relevance_scores.append(r_score)
        judge_reasoning.append(reasoning)

    except Exception as e:
        print(f"\nCritical Error on row {index}: {str(e)}")
        faithfulness_scores.append(0.0)
        relevance_scores.append(0.0)
        judge_reasoning.append(f"Judge Inference Error: {str(e)}")

df_results["judge_faithfulness"] = faithfulness_scores
df_results["judge_relevance"] = relevance_scores
df_results["judge_reasoning"] = judge_reasoning

df_results.to_csv("llm_judge_final_results.csv", index=False)
print("Finished! Saved to llm_judge_final_results.csv")

Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0 into memory...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Starting Local LLM Judge Evaluation for 200 answers...


  0%|          | 0/200 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Finished! Saved to llm_judge_final_results.csv


In [ ]:
import pandas as pd

def calculate_judge_metrics(file_path):
    df = pd.read_csv(file_path)

    if 'judge_faithfulness' not in df.columns or 'judge_relevance' not in df.columns:
        print("Error: Required columns not found in the dataset.")
        return

    faithfulness_avg = df['judge_faithfulness'].mean() * 100
    relevance_avg = df['judge_relevance'].mean() * 100

    faithfulness_out_of_5 = df['judge_faithfulness'].mean() * 5
    relevance_out_of_5 = df['judge_relevance'].mean() * 5

    print("="*40)
    print(" LLM JUDGE EVALUATION RESULTS ")
    print("="*40)
    print(f"Total Answers Graded: {len(df)}")
    print(f"Average Faithfulness: {faithfulness_avg:.2f}% ({faithfulness_out_of_5:.2f} / 5.0)")
    print(f"Average Relevance:    {relevance_avg:.2f}% ({relevance_out_of_5:.2f} / 5.0)")
    print("="*40)

calculate_judge_metrics('llm_judge_final_results (1).csv')

 LLM JUDGE EVALUATION RESULTS 
Total Answers Graded: 200
Average Faithfulness: 91.10% (4.55 / 5.0)
Average Relevance:    91.10% (4.55 / 5.0)


### Translation testing


In [ ]:
import pandas as pd

arabic_translations = [
    "ما هي النبرة العامة لتغطية الصحافة الإسبانية حول أبو سمبل في الستينيات؟", "ما هو الاسم الأصلي غير الملكي لرمسيس الأول؟",
    "ما هو الاسم المصري القديم لمدينة طيبة؟", "أين كانت تُحفظ تماثيل الكا؟", "ما هو الشيء المثير للاهتمام في الرصيف الذي يدعم تماثيل سخمت؟",
    "أين بنى توت عنخ آمون بفخر معبدين جديدين في النوبة؟", "أين يقع الموقع القديم لذراع أبو النجا؟", "ما هي زاوية انحدار الهرم؟",
    "ما هو أبو الهول العظيم؟", "كيف ينافس تصميم المعبد تصميم الكرنك الكوشي؟", "ما هي 'بحيرة خوفو'؟", "هل كانت النساء المصريات يحضرن المآدب الرسمية؟",
    "في أي سنوات زار بوركهارت وبلزوني المعابد؟", "هل تشارك النساء في الطقوس الموصوفة في البردية؟",
    "هل كان عيد سد (مهرجان سد) مدرجًا في التقويمات القياسية للمهرجانات المصرية القديمة؟", "من عمل مع دينون لإنشاء أول خريطة طوبوغرافية لهضبة الجيزة؟",
    "كيف نظر الجمهور إلى صور فريث في القرن التاسع عشر؟", "كم طالت فترة حكم الملك أمنحتب وفقًا ليوسيفوس؟",
    "زخارف أي من الملوك اللاحقين موثقة في معبد العساسيف؟", "ماذا كان يأمل رمسيس الثاني أن يحقق من خلال الاحتفال بالمهرجان 14 مرة؟",
    "ما هو متوسط حجم جعارين أمنحتب الثالث التذكارية؟", "ما هي السمة الجسدية المميزة على تمثال حتشبسوت المصنوع من الحجر الجيري المتبلور رغم الزي الذكوري؟",
    "ما هو مدى اتساع الجزء الأسطواني من قبة المعبد الكبير؟", "أي علامة قياس تشير إلى عيد ميلاد إيزيس؟", "ماذا كان يسمى قصر رمسيس؟",
    "ماذا استوردت لبناء السفن؟", "هل الجدران والأسقف في غرف المعبد خالية من الزخارف؟", "فيم كانت تستخدم قرية دير المدينة؟",
    "ما هي السمة المميزة لتماثيل أبو الهول في مدرسة تانيس؟", "ما هي القطعة من المعدات التي ربطت قضبان الرفع بخطاف الرافعة؟",
    "من قام بتوثيق الأهرامات والمقابر في مصر بين 1707 و 1726؟", "ما هو أدنى مستوى تم التنقيب عنه في ممر المقبرة 1152؟",
    "كم عدد الغرف الطويلة، التي يشار إليها غالباً باسم السرداب، والتي تفرعت غرباً من قاعة الأعمدة الأولى؟", "ماذا صور النقش الجداري على البوابة؟",
    "هل يمكن أن يكون أمنحتب الثالث والملكة تي أبناء عمومة من الدرجة الأولى؟", "لماذا من المرجح أن رمسيس الخامس والسادس استمرا في العمل في معبد العساسيف غير المكتمل؟",
    "هل أعلنت حتشبسوت نفسها حاكمة وحيدة فور وفاة تحتمس الثاني؟", "مما يتكون الكساء في قمة الهرم؟", "من نفذ الرسومات العديدة التي تحدد كل عملية قطع فردية؟",
    "كيف تختلف تقنية تشغيل الحجر الرومانية على الكتل المرممة عن التقنية الأصلية؟", "ما هو ارتفاع المدخل الغربي فوق الصحراء؟",
    "من أصبح حاكم عالم الموتى بعد قيامته؟", "هل كان النقش على القناع يسبق عصر توت عنخ آمون؟", "ما هو الفعل الذي يُنصح الطلاب بعدم القيام به أثناء استكشاف المعرض؟",
    "هل أبو الهول متماثل تمامًا؟", "من يتبع رمسيس الثاني وهو يقدم القرابين لمركبه المقدس؟", "هل استخدم البناؤون قياسات الذراع القياسية؟",
    "ما هي 'روح الآخ'؟", "هل يقدم متحف جيتي مواد تعليمية حول التصوير الفوتوغرافي المصري؟", "في العالم القديم، ماذا كان يحدث عادة للحكام غير الأكفاء؟",
    "من الذي يؤدي الحفل ويعتبر مشهدًا زخرفيًا رئيسيًا؟", "ما هي الوثيقة القديمة التي تشير إلى قصر مرتبط بمعبد؟", "ما هي المؤسسة التابعة للمشروع الأثري الأخير؟",
    "في أي عام حدث أول موكب غير رسمي للمومياوات؟", "من قام بالحفريات الأثرية في الفترة من 1842 إلى 1843؟", "ماذا درس مارك لينر للحصول على درجة الدكتوراه في جامعة ييل؟",
    "ما هي أهمية الإله آكر؟", "من هو المستكشف الأول الذي رسم بالخطأ المدخل السفلي يخرج من واجهة الهرم بدلاً من الفناء؟",
    "أي جزء من أبو الهول تم نحته من العضو الأول (Member I)؟", "هل تظهره لوحة سنفرو وهو يرتدي هذه التيجان؟",
    "لماذا لم يكن المصريون القدماء بشكل عام يؤلهون الكائنات الحية؟", "ماذا تعني كلمة 'حجري' (lithic)؟", "ما هي المرحلة التي تتضمنها إعادة البناء الافتراضية إلى جانب المراحل المصرية؟",
    "ما هي أهمية الشكل 2 في ورقة ريفز؟", "في المشهد الأول، ماذا يعد النص الموجود خلف تحوت الملك؟", "في الفن المصري التقليدي، ماذا كان يبرز اللون الأحمر بالنسبة لشخصيات الذكور؟",
    "من اكتشف 'بيت الكنز' داخل مجمع الرامسيوم؟", "إلى أين سافر إبراهيم خلال الفترة الانتقالية الأولى؟", "في أي إصدار متعدد الأجزاء نُشرت نتائج حملة نابليون؟",
    "ما هو تل العمارنة؟", "ما هو النقش البارز الملحوظ في السجل العلوي من الجدار الجنوبي؟", "لماذا سميت بغرفة الابتهالات؟",
    "ما هو المعلم الأثري الذي كان التركيز الرئيسي للمستكشفين الأوائل في الجيزة؟", "لماذا قام المصريون القدماء أحيانًا بتغيير اتجاه إنشاءاتهم؟",
    "ما هو عيد الوادي؟", "ما هي الآلة التي رفعت الكتل الحجرية من مواقعها الأصلية؟", "ما هو الفرق الجوهري في المواد بين الجرف الأصلي والتل الاصطناعي الجديد؟",
    "كيف نظر الجمهور إلى صور فريث في القرن التاسع عشر؟", "بأي نمط تم تشكيل الحواجب على التمثال؟", "لماذا يعتقد العلماء أن طقوس البردية الدرامية حدثت في معبد بدلاً من مقبرة؟",
    "ما هو 'بنتاؤور' فيما يتعلق بعمل المؤلف؟", "هل اقتبس اليونانيون أبو الهول من مصر؟", "كم عدد بنات رمسيس الثاني اللاتي صُورن في أبو سمبل؟",
    "كم عدد المداخل التي يمتلكها الهرم المُنحني؟", "كم عدد المقابر من الأسرة الثامنة عشرة إلى العشرين الموجودة في جميع الوديان مجتمعة؟",
    "كم عدد حفر القوارب الإجمالية التي اكتشفها حسن؟", "كم عدد الأطباق الضحلة الكبيرة ذات الحواف الحمراء التي وُجدت في الإيداع النمساوي؟",
    "ما نوع المعبد الذي كُرس لـ 'رع حوراختي' في الدير البحري؟", "كم كان متوقعًا أن يرتفع منسوب مياه النيل؟", "هل كان إحصاء الماشية يتم بشكل صارم كل سنتين خلال فترة حكم سنفرو؟",
    "ما هي الحالة غير الطبيعية التي أظهرتها الجمجمة؟", "لماذا يعتبر ندى الصباح مهماً للتفسير العلمي للصوت؟", "هل توجد حفرة قارب لا تزال مغطاة حتى اليوم؟",
    "هل وضع اليونانيون أبو الهول على عملاتهم النقدية؟", "من قام بتأليف الوثيقة بعنوان 'الرامسيوم (مصر)، البحوث الأثرية الحديثة'؟",
    "لماذا قام جمال عبد الناصر بتأميم قناة السويس؟", "ما هي 'ثكنات العمال'؟", "ما هي عناصر نموذج السفينة التي تم العثور عليها في STI-TR.To08؟",
    "هل اغتصب رمسيس الثاني تماثيل أبو الهول من ملوك سابقين؟", "إلى ماذا ترمز نماذج القوارب؟", "هل يحتوي المعبد الأصغر على تماثيل أوزيرية؟",
    "ما هو الهيكل الأسطوري السكندري الذي سقط بسبب نفس الزلزال؟", "ما هو معيار العرض والارتفاع للممر الهابط العلوي، والذي شوهد لأول مرة في الهرم الأحمر؟",
    "كم عدد الكتل التي تم توثيقها من المباني المحيطة في حشوة الأساس؟", "كم عدد قضبان الرفع التي كانت تستخدم عادة لكل كتلة؟",
    "كيف نقل المصريون الكتل الحجرية الضخمة؟", "ماذا كانت أسماء أبنائهم؟", "هل صحيح أن الرامسيوم تم بناؤه لملك؟", "ماذا كانت وظيفة المجمع الجنائزي؟",
    "هل اختلف عدد الأشياء من كل لون في الإيداع بشكل كبير؟", "كيف تختلف العربات ذات المظلات الخاصة برمسيس الثاني عن عربات توت عنخ آمون؟",
    "من كتب دراسة عن الكاهنات المعروفات باسم 'زوجات الإله'؟", "هل وُلد رمسيس الثاني من 'زوجة عظيمة'؟", "ماذا يجب أن يرتدي الزوار في الداخل؟",
    "كم يبلغ طول أبو الهول العظيم؟", "من عُثر عليه معها في المقبرة KV 60؟", "لماذا قام بحملة عسكرية في النوبة؟", "كيف تم تزيين وجه التابوت؟",
    "من كان رويو؟", "ما هو الدير القبطي المدمر الذي بُني فوق ملاذ روماني هنا؟", "ما هي عصابات الأنقاض غير المنتظمة بالقرب من قمة الهرم؟",
    "من هم 'أبناء حورس' في سياق الطقوس؟", "هل عمل مارك لينر على أبو الهول مؤخراً؟", "ما مدى ارتفاع المدخل العلوي؟",
    "ما هي التكنولوجيا الحديثة التي استخدمت مؤخراً لرسم خريطة محور الرامسيوم؟", "ما هي المملكة القديمة التي تُمثل البراعة الهندسية من خلال تمثالي ممنون؟",
    "ما هي السمة التي تجعل هرم خفرع مميزاً بصرياً في القمة؟", "لماذا تم وضع حجرة الدفن الرئيسية في الربع الشمالي الشرقي؟",
    "كيف يساعد تصميم المقبرة السياح على فهم الانتقال بعد فترة العمارنة؟", "لماذا اعتبر اقتراح الحاجز الخرساني من قبل سيرتويتس مزعجاً؟",
    "مم صُنعت اللحية المستعارة على التابوت؟", "ما هو الخطر الأكبر الذي واجهه كافيجليا ورجاله؟", "ما نوع المهارات الهندسية التي تمثلها التماثيل؟",
    "من لاحظ أن الجدار الشمالي لمعبد الوادي كان مائلاً بعض الشيء؟", "متى بُني أبو الهول؟", "نحو أي معلم طبيعي يتم توجيه الرامسيوم طوبوغرافيا؟",
    "من كتب مقال 'أبحاث جديدة في مقبرة رمسيس الثاني'؟", "ماذا تسمى اللوحة الحجرية الكبيرة على الجدار الجنوبي لواجهة المعبد الكبير؟",
    "ما نوع الطريق الذي يمتد من الهرم إلى المعبد الجنائزي؟", "هل نجح اللصوص في تجاوز البوابات المنزلقة في أهرامات أخرى؟",
    "من كان أول من نقب عن أبو الهول؟", "ما هو المسح المستوي؟", "هل بنى تحتمس الثالث معالم أثرية مثل حتشبسوت؟",
    "ما هو عرض تمثال أبو الهول الصغير والرقيق في الكرنك؟", "من كانت الملكة نفرتاري؟",
    "ماذا يقدم الملك لرمسيس الثاني المؤله على الدعامة الجنوبية لصرح عكاشة؟", "من استعرض الطبقات الجنائزية الفرعونية لجبانة طيبة في ورقة بحثية عام 2009؟",
    "ما هي الرؤية المسبقة في التفسير التأويلي؟", "هل عاش عامة الناس أم النخبة في أبيدوس خلال الفترة الانتقالية الأولى؟",
    "لماذا رفضت عنخ إسن آمون الزواج من الوزير الأكبر آي؟", "ما هو الغرض الرئيسي من 'الماميزي' أو المعبد الصغير؟",
    "متى أعادت AERA التنقيب في ثكنات العمال؟", "لمن يُنسب وجود أقدم الأدلة التاريخية للاحتفال بمهرجان سد؟", "هل تشققت القواعد قبل أم بعد وضع التماثيل؟",
    "من قاد الحملة إلى بونت؟", "ما هو الغرض من تميمة بيسيش-كاف؟", "ما هو طول الجدار المكتشف حديثاً من عصر الدولة القديمة؟",
    "ماذا يمثل رع حوراختي؟", "ما هي سنة حقوق الطبع والنشر الموجودة في الوثيقة التعليمية لمتحف جيتي؟", "كيف كان يوجد آتون في الأصل قبل حكم أخناتون؟",
    "هل يمكنك ذكر نص أدبي عُثر عليه في مجموعة برديات الرامسيوم؟", "ما هي الجامعة التي رعت المشروع الأثري لعام 2008 في المعبد؟",
    "هل نحت الفنانون المصريون الملامح بعمق للتفاعل مع ضوء النهار؟", "ماذا يرتدي تحتمس الثالث على اللوحة الرملية مع حتشبسوت؟",
    "ماذا يعني اسم 'سحتب إيب رع'؟", "ما هي 'عملية الكولوديون' التي استخدمها المصورون المصريون الأوائل؟", "ما هو القربان الذي يقدمه رمسيس الثالث في المشهد السابع؟",
    "ما هو البرنامج الذي استخدمه بوفال لحساب السمت الخاص بالمعبد؟", "ماذا دعم الأسطح الخرسانية الجديدة؟", "هل استمع أخناتون إلى ملك ميتاني؟",
    "ما هو بيت رع؟", "ما الذي أشار إلى وجود مقصورات جنائزية من الأسرة الثانية والعشرين في الممر المركزي؟", "ماذا يصور السجل المركزي لهذا الصرح؟",
    "من كان مسؤولاً إدارياً آخر ذُكر بجانب سننموت؟", "من أين بدأ الموكب الذهبي؟", "ما هو 'بير هاي'؟", "كيف تم تعزيز الكتل الحجرية قبل نقلها؟",
    "ما هي العوامل التي أثرت على أماكن وضع خطوط القطع؟", "ما هو اسم المصور التركي الذي التقط صوراً لأبو سمبل في أواخر القرن التاسع عشر؟",
    "ما هي الهياكل التي بُنيت مباشرة خلف كتل المعبد المعاد نصبها؟", "من أين نشأت الكتل الحجرية للهرم الأكبر؟", "هل سطح التمثال محفوظ بشكل جيد؟",
    "لمن يعود الطريق الصاعد الذي يشغل الجزء الجنوبي من معبد العساسيف جزئياً؟", "ماذا يرتكز على قاعدة طويلة في أقصى يسار الجدار الخلفي؟",
    "كم عدد القصائد التي كتبتها بالبيلا عن الزيارة؟", "في أي متحف يوجد التمثال النصفي المكسور مع الجعران؟", "إلى أي أسرة انتمى رمسيس الثاني؟",
    "من كان يعتبر والد رمسيس في طيبة؟", "ما هي أول غرفة كبيرة تدخلها داخل المعبد الرئيسي؟", "هل اكتشف هوارد كارتر مقبرة توت عنخ آمون قبل أم بعد عام 1920؟",
    "متى وقعت معركة قادش؟", "أي جزء من التمثال يحتوي على غالبية الكتابات اليونانية؟", "ماذا وُجد تحت طبقة الصنادل في أحد النعوش؟",
    "لماذا كان يجب تصوير رمسيس الثاني جالساً بينما وقفت إيزيس في الفناء الثاني؟", "ما الذي يحمي الملك طوال حفل التتويج؟",
    "من كتب المقال الرائد لعام 2012 الذي يجادل بأن المظلة تنتمي إلى العربة؟", "هل يمكنك تسمية بعض أعضاء الفريق المصري الآخرين الذين ساعدوا في الاكتشاف؟",
    "من كان 'الملك المحارب'؟", "متى فُتحت مقبرة نفرتاري (QV 66) لأول مرة للزوار قبل إغلاقها في منتصف الثلاثينيات؟", "على أي ضفة من النيل نحتت المعابد في الأصل؟"
]

df_gt = pd.read_csv("corrected_ground_truth_final - ground_truth_final.csv (1).csv")
english_queries = df_gt['query'].tolist()

df_test_set = pd.DataFrame({
    'arabic_query': arabic_translations,
    'expected_english_reference': english_queries
})

df_test_set.to_csv('multilingual_translation_test_dataset.csv', index=False)
print("Successfully generated: multilingual_translation_test_dataset.csv with 200 authentic translation pairs!")

Successfully generated: multilingual_translation_test_dataset.csv with 200 authentic translation pairs!


In [ ]:
!pip install evaluate sacrebleu bert-score rouge-score deep-translator torch

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.5 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=5c0e1db1d8ad3af24d23785aa4abcc1a1414f487ae0c070f7a77806f803b7600
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [ ]:
import pandas as pd
import evaluate
from tqdm.auto import tqdm
from deep_translator import GoogleTranslator

df_test = pd.read_csv("multilingual_translation_test_dataset.csv")

print("Loading Academic Metrics...")
bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")
bert_metric = evaluate.load("bertscore")

predictions = []
print(f"Translating {len(df_test)} queries using pipeline...")
translator = GoogleTranslator(source='ar', target='en')

for text in tqdm(df_test["arabic_query"]):
    try:
        translated_text = translator.translate(text)
        predictions.append(translated_text)
    except Exception as e:
        predictions.append("")

df_test["model_translation"] = predictions

print("\nCalculating academic scores...")

references_formatted = [[ref] for ref in df_test["expected_english_reference"].tolist()]
flat_references = df_test["expected_english_reference"].tolist()

bleu_score = bleu_metric.compute(predictions=predictions, references=references_formatted)

chrf_score = chrf_metric.compute(predictions=predictions, references=references_formatted, word_order=2)

bert_score = bert_metric.compute(predictions=predictions, references=flat_references, lang="en", model_type="roberta-large")
avg_bert = sum(bert_score["f1"]) / len(bert_score["f1"]) * 100

print("\n" + "="*50)
print("FINAL PIPELINE TRANSLATION METRICS")
print("="*50)
print(f"Total Test Queries: {len(df_test)}")
print(f"1. sacreBLEU Score (Word level):       {bleu_score['score']:.2f} / 100")
print(f"2. chrF++ Score (Morphology level):    {chrf_score['score']:.2f} / 100")
print(f"3. BERTScore F1 (Semantic Meaning):    {avg_bert:.2f}%")
print("="*50)

df_test.to_csv("final_translation_evaluation_log.csv", index=False)

Loading Academic Metrics...


Translating 200 queries using pipeline...


  0%|          | 0/200 [00:00<?, ?it/s]


Calculating academic scores...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



FINAL PIPELINE TRANSLATION METRICS
Total Test Queries: 200
1. sacreBLEU Score (Word level):       50.74 / 100
2. chrF++ Score (Morphology level):    73.13 / 100
3. BERTScore F1 (Semantic Meaning):    97.18%
